# 03 — Sentinel-2 Exploration and Vegetation Features

This notebook defines the Sentinel-2 component of the wildfire-risk dataset for Mainland Portugal.

The modelling unit is:

> **one 5 × 5 km grid cell × one weekly reference date (Monday)**

For every `reference_date`, all satellite information must be available **strictly before that Monday**. The target week is the Monday–Sunday interval beginning on the reference date.

The final workflow developed here solves four practical problems:

1. Sentinel-2 acquisition frequency varies spatially and temporally;
2. one acquisition day can contain several overlapping MGRS tiles;
3. clouds, missing pixels and incomplete footprints can make recent imagery unusable;
4. vegetation indices should be evaluated only where fire-relevant vegetation exists.

The final tabular Sentinel features are:

- static vegetation support: `vegetation_fraction`;
- observation metadata: `selected_date`, `image_age_days`, `valid_pct`;
- current vegetation state: NDVI and NDMI mean, median, P10 and P90;
- recent vegetation dynamics: `vegetation_obs_60d`, `NDVI_slope_30d`, `NDMI_slope_30d`.

The final weekly image-selection policy is frozen at **95% / 90% / 85%** valid vegetation coverage over 7 / 14 / 30-day look-back windows.

## 1. Setup and data sources

Google Earth Engine provides Sentinel-2 Surface Reflectance imagery. The local 5 × 5 km grid is the same grid created during the geospatial preprocessing stage.

The notebook keeps the exploratory and methodological code required to justify the final decisions, but removes repeated experiments and failed performance optimizations.

In [ ]:
from pathlib import Path
import time

import ee
import geopandas as gpd
import pandas as pd

In [ ]:
ee.Authenticate(auth_mode="localhost")
ee.Initialize(project="wildfireproject-506722")

In [ ]:
GRID_PATH = Path("../data/interim/grid_5km_portugal.parquet")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

grid = gpd.read_parquet(GRID_PATH)
grid_wgs84 = grid.to_crs("EPSG:4326")

sentinel2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")


def get_cell_ee_geometry(cell_id):
    row = grid_wgs84.loc[
        grid_wgs84["cell_id"] == cell_id,
        "geometry"
    ]

    if row.empty:
        raise ValueError(f"Unknown cell_id: {cell_id}")

    return ee.Geometry(row.iloc[0].__geo_interface__)

### 1.1 Sentinel-2 source

The analysis uses `COPERNICUS/S2_SR_HARMONIZED`.

- `SCL` is used to measure local observation quality.
- `B4` and `B8` are used for NDVI.
- `B8` and `B11` are used for NDMI.
- NDVI is aggregated at 10 m and NDMI at 20 m.

The nominal revisit interval is not treated as the effective observation interval for a grid cell; actual acquisition days are derived from the collection itself.

## 2. Local Sentinel-2 observation quality

Scene-level `CLOUDY_PIXEL_PERCENTAGE` is not sufficient for this project because it describes the complete Sentinel scene, not the target 5 × 5 km cell.

Exploration of `PT_0008` in May–October 2019 found:

- 72 Sentinel-2 objects / acquisition days;
- most consecutive acquisition gaps were 2–3 days;
- scene-level cloudiness had mean ≈ 38.15% and median ≈ 32.61%;
- scene cloudiness and local valid area were correlated, but several globally cloudy scenes still provided a mostly clear target cell.

Therefore, image selection is based on **pixel-level validity inside the target region**, not scene-level cloud metadata.

### 2.1 Scene Classification Layer (SCL)

The following SCL classes are considered invalid:

- `0` — No Data
- `1` — Saturated or defective
- `3` — Cloud shadow
- `8` — Cloud medium probability
- `9` — Cloud high probability
- `10` — Thin cirrus
- `11` — Snow or ice

Pixels outside the image footprint are explicitly converted to zero with `unmask(0)`. This prevents partially observed cells from appearing artificially perfect because missing pixels disappeared from the denominator.

In [ ]:
def scl_to_valid_mask(image):
    scl = image.select("SCL")

    valid = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
        .rename("valid")
    )

    return valid.unmask(0)


def get_acquisition_days(collection):
    timestamps = collection.aggregate_array(
        "system:time_start"
    ).getInfo()

    if not timestamps:
        return pd.Series(dtype="datetime64[ns]")

    dates = pd.to_datetime(
        timestamps,
        unit="ms"
    )

    return (
        pd.Series(dates)
        .dt.normalize()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )


def build_daily_valid_mask(
    collection,
    acquisition_day
):
    start_date = ee.Date(
        pd.Timestamp(acquisition_day).strftime("%Y-%m-%d")
    )
    end_date = start_date.advance(1, "day")

    daily_images = collection.filterDate(
        start_date,
        end_date
    )

    return (
        daily_images
        .map(scl_to_valid_mask)
        .max()
        .rename("valid")
    )


def get_daily_quality_table(
    collection,
    geometry
):
    acquisition_days = get_acquisition_days(
        collection
    )

    if len(acquisition_days) == 0:
        return pd.DataFrame(
            columns=["date", "valid_pct"]
        )

    features = []

    for day in acquisition_days:
        daily_valid = build_daily_valid_mask(
            collection,
            day
        )

        valid_fraction = daily_valid.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geometry,
            scale=20,
            maxPixels=100000
        ).get("valid")

        features.append(
            ee.Feature(
                None,
                {
                    "date": day.strftime("%Y-%m-%d"),
                    "valid_fraction": valid_fraction
                }
            )
        )

    result = ee.FeatureCollection(
        features
    ).getInfo()

    quality = pd.DataFrame(
        [
            feature["properties"]
            for feature in result["features"]
        ]
    )

    quality["date"] = pd.to_datetime(
        quality["date"]
    )
    quality["valid_pct"] = (
        quality["valid_fraction"] * 100
    )

    return (
        quality[["date", "valid_pct"]]
        .sort_values("date")
        .reset_index(drop=True)
    )

### 2.2 Acquisition day is the logical observation unit

A cell can intersect several Sentinel-2 MGRS tiles on the same physical acquisition day.

For `PT_2093` in May–October 2019:

- Earth Engine returned **288 image objects**;
- these represented only **72 unique acquisition days**;
- every acquisition day was represented by four overlapping MGRS tiles.

The logical unit is therefore:

> **cell × acquisition day**, not cell × tile object.

For observation quality, each tile is converted to a binary valid mask first. The daily masks are then combined with a pixel-wise maximum, so a clear observation from any overlapping tile can fill the pixel.

In [ ]:
# Optional diagnostic reproducing the MGRS-overlap example.

tile_test_geometry = get_cell_ee_geometry("PT_2093")

tile_test_collection = (
    sentinel2
    .filterBounds(tile_test_geometry)
    .filterDate("2019-05-01", "2019-11-01")
)

tile_timestamps = pd.to_datetime(
    tile_test_collection
    .aggregate_array("system:time_start")
    .getInfo(),
    unit="ms"
)

tile_days = pd.Series(
    tile_timestamps
).dt.normalize()

print(
    "Earth Engine objects:",
    tile_test_collection.size().getInfo()
)
print(
    "Unique acquisition days:",
    tile_days.nunique()
)

### 2.3 Incomplete footprint correction

A methodological bug was detected during validation. The original mean ignored masked pixels, so an acquisition observing only part of a cell could be reported as 100% valid.

For `PT_2087` on 2017-04-23:

- old implementation: approximately 100% valid;
- actually observed cell area: **23.65%**;
- corrected implementation: approximately **23.65% valid**.

The `unmask(0)` logic above is therefore essential and is retained in the final workflow.

## 3. Static vegetation-support mask

Observation quality for vegetation features should describe the pixels where NDVI/NDMI are meaningful for wildfire fuel condition, rather than the complete square cell.

A static support mask is built from **CORINE Land Cover 2018** (`COPERNICUS/CORINE/V20/100m/2018`).

Included classes cover:

- arable and permanent agriculture;
- pastures and heterogeneous agricultural areas;
- agroforestry;
- broad-leaved, coniferous and mixed forest;
- natural grassland, heathland, sclerophyllous vegetation and transitional woodland/shrub.

Artificial surfaces, predominantly bare land, wetlands and water are excluded.

CORINE is used only to define **where** vegetation indices are meaningful; it is not used as a dynamic model feature.

In [ ]:
clc2018 = ee.Image(
    "COPERNICUS/CORINE/V20/100m/2018"
).select("landcover")

VEGETATION_CLC_CLASSES = [
    211, 212, 213,
    221, 222, 223,
    231,
    241, 242, 243, 244,
    311, 312, 313,
    321, 322, 323, 324
]

vegetation_mask = (
    clc2018
    .remap(
        VEGETATION_CLC_CLASSES,
        [1] * len(VEGETATION_CLC_CLASSES),
        0
    )
    .rename("vegetation_support")
)

In [ ]:
def get_vegetation_fraction(cell_id):
    geometry = get_cell_ee_geometry(cell_id)

    fraction = vegetation_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geometry,
        scale=100,
        maxPixels=100000
    ).get("vegetation_support")

    value = fraction.getInfo()

    return 0 if value is None else value


def get_daily_vegetation_quality_table(
    collection,
    geometry,
    vegetation_mask
):
    acquisition_days = get_acquisition_days(
        collection
    )

    if len(acquisition_days) == 0:
        return pd.DataFrame(
            columns=["date", "valid_pct"]
        )

    features = []

    for day in acquisition_days:
        daily_valid = build_daily_valid_mask(
            collection,
            day
        )

        valid_on_vegetation = (
            daily_valid
            .updateMask(vegetation_mask)
        )

        valid_fraction = valid_on_vegetation.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geometry,
            scale=20,
            maxPixels=100000
        ).get("valid")

        features.append(
            ee.Feature(
                None,
                {
                    "date": day.strftime("%Y-%m-%d"),
                    "valid_fraction": valid_fraction
                }
            )
        )

    result = ee.FeatureCollection(
        features
    ).getInfo()

    quality = pd.DataFrame(
        [
            feature["properties"]
            for feature in result["features"]
        ]
    )

    quality["date"] = pd.to_datetime(
        quality["date"]
    )
    quality["valid_pct"] = (
        quality["valid_fraction"]
        .fillna(0)
        * 100
    )

    return (
        quality[["date", "valid_pct"]]
        .sort_values("date")
        .reset_index(drop=True)
    )

### 3.1 Sanity checks

The CORINE support mask behaved correctly at contrasting locations:

| Cell | Vegetation support |
|---|---:|
| `PT_2933` | **99.95%** |
| `PT_2943` | **39.38%** |
| `PT_1832` | **23.23%** |
| `PT_2093` | **0%** |

The denominator can materially change which image is selected. For `PT_2943`:

| Date | Full-cell valid | Vegetation valid |
|---|---:|---:|
| 2019-04-30 | 94.90% | 87.24% |
| 2019-05-05 | 79.80% | 89.14% |

For the 2019-05-06 reference date, the vegetation-based policy therefore selects **2019-05-05** rather than 2019-04-30.

This confirms that quality should be measured over the same static vegetation support used to derive NDVI/NDMI.

## 4. Weekly adaptive image-selection policy

For every Monday reference date, the algorithm selects the **most recent qualifying acquisition strictly before the reference date**.

The quality requirement becomes less strict as the look-back window grows. This balances:

- recency;
- observation quality;
- weekly spatial coverage.

No wildfire target information is used to choose these thresholds.

In [ ]:
def make_reference_dates(year):
    dates = pd.date_range(
        start=f"{year}-05-01",
        end=f"{year}-10-31",
        freq="W-MON"
    )

    return dates[
        dates + pd.Timedelta(days=6)
        <= pd.Timestamp(f"{year}-10-31")
    ]


def select_adaptive_image(
    quality,
    reference_date,
    rules
):
    reference_date = pd.Timestamp(
        reference_date
    )

    for window_days, min_valid_pct in rules:
        window_start = (
            reference_date
            - pd.Timedelta(days=window_days)
        )

        candidates = quality[
            (quality["date"] >= window_start)
            & (quality["date"] < reference_date)
            & (
                quality["valid_pct"]
                >= min_valid_pct
            )
        ]

        if not candidates.empty:
            selected = (
                candidates
                .sort_values("date")
                .iloc[-1]
            )

            return {
                "selected_date": selected["date"],
                "valid_pct": selected["valid_pct"],
                "age_days": (
                    reference_date
                    - selected["date"]
                ).days,
                "window_used": window_days,
                "threshold_used": min_valid_pct
            }

    return {
        "selected_date": pd.NaT,
        "valid_pct": None,
        "age_days": None,
        "window_used": None,
        "threshold_used": None
    }

### 4.1 Frozen development / holdout sample

A separate spatial validation sample was used instead of tuning on the original pilot cells:

- 5 latitude strata × 4 longitude strata = 20 spatial strata;
- 2 cells sampled per stratum;
- 40 cells total;
- one cell per stratum assigned to development and one to holdout;
- pilot cells excluded;
- fixed split retained throughout threshold selection.

All 40 sampled validation cells had `land_fraction = 1.0`, but their CORINE vegetation fractions ranged from approximately 23% to 100%.

In [ ]:
development_cell_ids = [
    "PT_0670", "PT_0983", "PT_2331", "PT_2967",
    "PT_0899", "PT_1244", "PT_2684", "PT_3328",
    "PT_0316", "PT_1832", "PT_2267", "PT_3102",
    "PT_0941", "PT_1517", "PT_1963", "PT_3282",
    "PT_0848", "PT_1196", "PT_2087", "PT_3672"
]

holdout_cell_ids = [
    "PT_1570", "PT_0889", "PT_1965", "PT_1789",
    "PT_0716", "PT_1538", "PT_2661", "PT_2943",
    "PT_0266", "PT_2685", "PT_1853", "PT_3075",
    "PT_2927", "PT_2796", "PT_2280", "PT_0377",
    "PT_0642", "PT_1033", "PT_3240", "PT_1912"
]

validation_cell_ids = (
    development_cell_ids
    + holdout_cell_ids
)

print("Development:", len(development_cell_ids))
print("Holdout:", len(holdout_cell_ids))

### 4.2 Candidate-rule evaluation

Twenty-nine monotonic threshold cascades were compared on the **development cells only**.

Objectives:

1. maximize weekly coverage;
2. minimize P95 image age;
3. maximize P05 selected valid percentage.

The target variable was not used. The code below retains the evaluation logic; the expensive Earth Engine quality tables do not need to be recomputed merely to inspect the recorded methodological result.

In [ ]:
candidate_rules = []

for threshold_7d in [85, 90, 95]:
    for threshold_14d in [80, 85, 90]:
        for threshold_30d in [70, 75, 80, 85]:
            if (
                threshold_7d
                >= threshold_14d
                >= threshold_30d
            ):
                candidate_rules.append(
                    (
                        threshold_7d,
                        threshold_14d,
                        threshold_30d
                    )
                )

print("Candidate rules:", len(candidate_rules))


def evaluate_rule(
    thresholds,
    quality_dict,
    cell_ids,
    reference_dates
):
    t7, t14, t30 = thresholds

    rules = (
        (7, t7),
        (14, t14),
        (30, t30)
    )

    rows = []

    for cell_id in cell_ids:
        quality = quality_dict[cell_id]

        for reference_date in reference_dates:
            selection = select_adaptive_image(
                quality,
                reference_date,
                rules=rules
            )

            rows.append({
                "cell_id": cell_id,
                "reference_date": reference_date,
                **selection
            })

    rows = pd.DataFrame(rows)
    selected = rows[
        rows["selected_date"].notna()
    ]

    return {
        "thresholds": thresholds,
        "n_total": len(rows),
        "n_selected": len(selected),
        "n_missing": rows[
            "selected_date"
        ].isna().sum(),
        "coverage_pct": (
            len(selected) / len(rows) * 100
        ),
        "mean_age": selected[
            "age_days"
        ].mean(),
        "p95_age": selected[
            "age_days"
        ].quantile(0.95),
        "p05_valid_pct": selected[
            "valid_pct"
        ].quantile(0.05),
        "min_valid_pct": selected[
            "valid_pct"
        ].min()
    }


def pareto_frontier(results):
    rows = []

    for i, row_i in results.iterrows():
        dominated = False

        for j, row_j in results.iterrows():
            if i == j:
                continue

            at_least_as_good = (
                row_j["coverage_pct"]
                >= row_i["coverage_pct"]
                and row_j["p95_age"]
                <= row_i["p95_age"]
                and row_j["p05_valid_pct"]
                >= row_i["p05_valid_pct"]
            )

            strictly_better = (
                row_j["coverage_pct"]
                > row_i["coverage_pct"]
                or row_j["p95_age"]
                < row_i["p95_age"]
                or row_j["p05_valid_pct"]
                > row_i["p05_valid_pct"]
            )

            if at_least_as_good and strictly_better:
                dominated = True
                break

        if not dominated:
            rows.append(row_i)

    return pd.DataFrame(rows)

### 4.3 Final development result

The combined 2017–2018 development set contained **1020 cell-weeks**.

The final Pareto frontier reduced to two meaningful alternatives:

| Rule (7d / 14d / 30d) | Coverage | Mean age | P95 age | P05 valid | Minimum valid |
|---|---:|---:|---:|---:|---:|
| 95 / 90 / 75 | 97.84% | 5.94 d | 20 d | 94.86% | 75.72% |
| **95 / 90 / 85** | **97.45%** | 6.02 d | **20 d** | **95.71%** | **85.67%** |

The 85% fallback sacrificed only **4 cell-weeks out of 1020** relative to the 75% fallback while eliminating the low-quality tail below 85%.

The final rule was therefore frozen at **95 / 90 / 85** before evaluating the holdout cells.

In [ ]:
FINAL_RULE = (
    (7, 95),
    (14, 90),
    (30, 85)
)

### 4.4 Independent holdout validation

After freezing the rule, no further threshold tuning was performed.

| Metric | 2017 spatial holdout | 2018 spatial + temporal holdout |
|---|---:|---:|
| Cell-weeks | 520 | 500 |
| Selected | 501 | 494 |
| Missing | 19 | 6 |
| Coverage | **96.35%** | **98.80%** |
| Mean image age | 6.11 d | 5.84 d |
| P95 image age | 20 d | 20 d |
| P05 valid vegetation | **96.08%** | **96.33%** |
| Minimum valid vegetation | **86.79%** | **86.23%** |

The rule generalized without a collapse in coverage, recency or lower-tail quality.

### Final weekly observation policy

For each `reference_date`:

```text
0–7 days   → valid vegetation coverage >= 95%
0–14 days  → valid vegetation coverage >= 90%
0–30 days  → valid vegetation coverage >= 85%
otherwise  → satellite observation is missing
```

Within each level, choose the **most recent** qualifying acquisition.

Retain `image_age_days` and `valid_pct` as metadata in the final dataset.

## 5. Cloud-masked reflectance and vegetation indices

Image selection identifies the acquisition day; feature extraction then reconstructs the spectral image correctly.

Each Sentinel tile is masked **before** mosaicking. This allows a clear overlapping tile to fill a pixel that is invalid in another tile.

The static CORINE vegetation mask is then applied to NDVI and NDMI.

In [ ]:
def mask_sentinel_reflectance(image):
    valid_mask = scl_to_valid_mask(
        image
    )

    return image.updateMask(
        valid_mask
    )


def build_daily_reflectance_mosaic(
    collection,
    acquisition_day
):
    start_date = ee.Date(
        pd.Timestamp(
            acquisition_day
        ).strftime("%Y-%m-%d")
    )
    end_date = start_date.advance(
        1,
        "day"
    )

    daily_images = collection.filterDate(
        start_date,
        end_date
    )

    masked_images = daily_images.map(
        mask_sentinel_reflectance
    )

    return masked_images.mosaic()

### 5.1 Current-state statistics

For every selected acquisition, the final tabular representation contains:

**NDVI (`B8`, `B4`; 10 m)**  
`NDVI_mean`, `NDVI_median`, `NDVI_p10`, `NDVI_p90`

**NDMI (`B8`, `B11`; 20 m)**  
`NDMI_mean`, `NDMI_median`, `NDMI_p10`, `NDMI_p90`

Using multiple distribution summaries preserves some within-cell heterogeneity without storing the full image for the tabular model.

In [ ]:
def extract_acquisition_stats_batch(
    collection,
    geometry,
    vegetation_mask,
    good_quality
):
    if good_quality.empty:
        return pd.DataFrame()

    reducer = (
        ee.Reducer.mean()
        .combine(
            reducer2=ee.Reducer.median(),
            sharedInputs=True
        )
        .combine(
            reducer2=ee.Reducer.percentile(
                [10, 90]
            ),
            sharedInputs=True
        )
    )

    features = []

    for acquisition_date in good_quality["date"]:
        reflectance = (
            build_daily_reflectance_mosaic(
                collection,
                acquisition_date
            )
        )

        ndvi = (
            reflectance
            .normalizedDifference(["B8", "B4"])
            .rename("NDVI")
            .updateMask(vegetation_mask)
        )

        ndmi = (
            reflectance
            .normalizedDifference(["B8", "B11"])
            .rename("NDMI")
            .updateMask(vegetation_mask)
        )

        ndvi_stats = ndvi.reduceRegion(
            reducer=reducer,
            geometry=geometry,
            scale=10,
            maxPixels=1000000
        )

        ndmi_stats = ndmi.reduceRegion(
            reducer=reducer,
            geometry=geometry,
            scale=20,
            maxPixels=1000000
        )

        properties = (
            ee.Dictionary(ndvi_stats)
            .combine(
                ee.Dictionary(ndmi_stats),
                overwrite=True
            )
            .set(
                "selected_date",
                pd.Timestamp(
                    acquisition_date
                ).strftime("%Y-%m-%d")
            )
        )

        features.append(
            ee.Feature(None, properties)
        )

    # Evaluate all acquisition statistics in one Earth Engine request.
    result = ee.FeatureCollection(
        features
    ).getInfo()

    stats = pd.DataFrame(
        [
            feature["properties"]
            for feature in result["features"]
        ]
    )

    stats["selected_date"] = pd.to_datetime(
        stats["selected_date"]
    )

    quality_lookup = (
        good_quality[
            ["date", "valid_pct"]
        ]
        .rename(
            columns={"date": "selected_date"}
        )
    )

    return (
        stats
        .merge(
            quality_lookup,
            on="selected_date",
            how="left"
        )
        .sort_values("selected_date")
        .reset_index(drop=True)
    )

### 5.2 60-day vegetation trend

Weekly rows can reuse the same Sentinel acquisition. Trend estimation must therefore use **distinct good acquisition days**, not repeated weekly rows.

Final trend definition:

- look back 60 days from `reference_date`;
- use every distinct acquisition with `valid_pct >= 85%`;
- require at least 3 observations;
- regress NDVI / NDMI against elapsed days;
- normalize the slope to a 30-day change.

For `PT_2933` at `2019-07-15`, the validated example produced:

- `vegetation_obs_60d = 4`
- `NDVI_slope_30d = -0.06137`
- `NDMI_slope_30d = -0.09045`

The negative slopes correctly summarize the strong early-summer drying visible in the acquisition series.

In [ ]:
def calculate_vegetation_trend(
    acquisition_series,
    reference_date,
    window_days=60,
    min_observations=3
):
    reference_date = pd.Timestamp(
        reference_date
    )
    window_start = (
        reference_date
        - pd.Timedelta(days=window_days)
    )

    trend_data = (
        acquisition_series[
            (
                acquisition_series["selected_date"]
                >= window_start
            )
            & (
                acquisition_series["selected_date"]
                < reference_date
            )
        ]
        .drop_duplicates(
            subset="selected_date"
        )
        .sort_values("selected_date")
        .copy()
    )

    n_observations = len(
        trend_data
    )

    if n_observations < min_observations:
        return {
            "vegetation_obs_60d": n_observations,
            "NDVI_slope_30d": None,
            "NDMI_slope_30d": None
        }

    trend_data["days"] = (
        trend_data["selected_date"]
        - trend_data["selected_date"].min()
    ).dt.days

    ndvi_slope_daily = (
        trend_data["NDVI_mean"].cov(
            trend_data["days"]
        )
        / trend_data["days"].var()
    )

    ndmi_slope_daily = (
        trend_data["NDMI_mean"].cov(
            trend_data["days"]
        )
        / trend_data["days"].var()
    )

    return {
        "vegetation_obs_60d": n_observations,
        "NDVI_slope_30d": ndvi_slope_daily * 30,
        "NDMI_slope_30d": ndmi_slope_daily * 30
    }

## 6. Efficient `cell × year` feature pipeline

A correct but naive implementation recalculated NDVI/NDMI repeatedly for every weekly row. That is unnecessary because several weeks can reuse the same acquisition.

The final processing unit is:

> **one cell × one year**

For each cell-year:

1. calculate daily vegetation-quality once;
2. retain all acquisition days with `valid_pct >= 85%`;
3. calculate NDVI/NDMI once per good acquisition in an Earth Engine batch;
4. generate every weekly row locally from that acquisition table.

This preserves the exact methodology while avoiding repeated Earth Engine round-trips.

In [ ]:
def build_cell_year_acquisition_table(
    cell_id,
    year,
    vegetation_fraction=None
):
    geometry = get_cell_ee_geometry(
        cell_id
    )

    if vegetation_fraction is None:
        vegetation_fraction = (
            get_vegetation_fraction(
                cell_id
            )
        )

    if vegetation_fraction <= 0:
        return (
            pd.DataFrame(),
            vegetation_fraction
        )

    reference_dates = (
        make_reference_dates(year)
    )

    start_date = (
        min(reference_dates)
        - pd.Timedelta(days=60)
    )
    end_date = max(
        reference_dates
    )

    collection = (
        sentinel2
        .filterBounds(geometry)
        .filterDate(
            start_date.strftime("%Y-%m-%d"),
            end_date.strftime("%Y-%m-%d")
        )
    )

    quality = (
        get_daily_vegetation_quality_table(
            collection,
            geometry,
            vegetation_mask
        )
    )

    good_quality = (
        quality[
            quality["valid_pct"] >= 85
        ]
        .copy()
        .reset_index(drop=True)
    )

    acquisition_table = (
        extract_acquisition_stats_batch(
            collection,
            geometry,
            vegetation_mask,
            good_quality
        )
    )

    return (
        acquisition_table,
        vegetation_fraction
    )

In [ ]:
def build_cell_year_weekly_features(
    cell_id,
    year,
    acquisition_table,
    vegetation_fraction
):
    weekly_rows = []

    if acquisition_table.empty:
        for reference_date in make_reference_dates(
            year
        ):
            weekly_rows.append({
                "cell_id": cell_id,
                "reference_date": reference_date,
                "vegetation_fraction": vegetation_fraction,
                "selected_date": pd.NaT,
                "image_age_days": None,
                "valid_pct": None,
                "NDVI_mean": None,
                "NDVI_median": None,
                "NDVI_p10": None,
                "NDVI_p90": None,
                "NDMI_mean": None,
                "NDMI_median": None,
                "NDMI_p10": None,
                "NDMI_p90": None,
                "vegetation_obs_60d": 0,
                "NDVI_slope_30d": None,
                "NDMI_slope_30d": None
            })

        return pd.DataFrame(
            weekly_rows
        )

    quality_table = (
        acquisition_table[
            ["selected_date", "valid_pct"]
        ]
        .rename(
            columns={"selected_date": "date"}
        )
    )

    for reference_date in make_reference_dates(
        year
    ):
        selection = select_adaptive_image(
            quality_table,
            reference_date,
            rules=FINAL_RULE
        )

        current_stats = {
            "NDVI_mean": None,
            "NDVI_median": None,
            "NDVI_p10": None,
            "NDVI_p90": None,
            "NDMI_mean": None,
            "NDMI_median": None,
            "NDMI_p10": None,
            "NDMI_p90": None
        }

        if not pd.isna(
            selection["selected_date"]
        ):
            selected_row = acquisition_table[
                acquisition_table[
                    "selected_date"
                ]
                == selection["selected_date"]
            ].iloc[0]

            current_stats = {
                "NDVI_mean": selected_row["NDVI_mean"],
                "NDVI_median": selected_row["NDVI_median"],
                "NDVI_p10": selected_row["NDVI_p10"],
                "NDVI_p90": selected_row["NDVI_p90"],
                "NDMI_mean": selected_row["NDMI_mean"],
                "NDMI_median": selected_row["NDMI_median"],
                "NDMI_p10": selected_row["NDMI_p10"],
                "NDMI_p90": selected_row["NDMI_p90"]
            }

        trend = calculate_vegetation_trend(
            acquisition_table,
            reference_date
        )

        weekly_rows.append({
            "cell_id": cell_id,
            "reference_date": reference_date,
            "vegetation_fraction": vegetation_fraction,
            "selected_date": selection["selected_date"],
            "image_age_days": selection["age_days"],
            "valid_pct": selection["valid_pct"],
            **current_stats,
            **trend
        })

    return pd.DataFrame(
        weekly_rows
    )


def process_cell_year_final(
    cell_id,
    year,
    vegetation_fraction
):
    acquisitions, _ = (
        build_cell_year_acquisition_table(
            cell_id,
            year,
            vegetation_fraction=vegetation_fraction
        )
    )

    return build_cell_year_weekly_features(
        cell_id,
        year,
        acquisitions,
        vegetation_fraction
    )

### 6.1 Functional validation

Three contrasting 2019 cells were used as final sanity checks:

**`PT_2933` — ≈99.95% vegetation support**
- 22 good acquisitions;
- 25 weekly rows;
- for 2019-07-15: selected 2019-07-14, age 1 day, 100% valid;
- NDVI mean ≈ 0.4416, NDMI mean ≈ -0.1092;
- trend slopes matched the manually validated calculations.

**`PT_2943` — ≈39.38% vegetation support**
- 24 good acquisitions;
- vegetation-based quality correctly changes the early-May image selection.

**`PT_2093` — 0% vegetation support**
- 25 weekly rows are preserved;
- Sentinel vegetation features are missing rather than inventing values.

This validates the pipeline at high, intermediate and zero vegetation support.

In [ ]:
# Optional smoke test. This makes Earth Engine calls.

test_cell = "PT_2087"
test_year = 2019

test_vegetation_fraction = (
    get_vegetation_fraction(
        test_cell
    )
)

test_weekly = process_cell_year_final(
    test_cell,
    test_year,
    test_vegetation_fraction
)

print(test_weekly.shape)
test_weekly.head()

## 7. Static vegetation fraction for the full grid

`vegetation_fraction` is static and should be computed only once per cell.

The national calculation over all 3822 grid cells took approximately 26 seconds in the validated run. Its distribution was:

- cells: **3822**
- zero vegetation support: **14**
- vegetation support > 0: **3808**
- mean: **0.9028**
- median: **0.9822**
- 25th percentile: **0.9201**
- 75th percentile: **1.0000**

Most cells therefore require Sentinel processing; skipping zero-vegetation cells alone does not materially reduce the production workload.

In [ ]:
def calculate_grid_vegetation_fraction(
    grid_wgs84,
    vegetation_mask
):
    features = []

    for _, row in grid_wgs84.iterrows():
        features.append(
            ee.Feature(
                ee.Geometry(
                    row.geometry.__geo_interface__
                ),
                {
                    "cell_id": row["cell_id"]
                }
            )
        )

    cells_fc = ee.FeatureCollection(
        features
    )

    result = vegetation_mask.reduceRegions(
        collection=cells_fc,
        reducer=ee.Reducer.mean(),
        scale=100
    ).getInfo()

    rows = []

    for feature in result["features"]:
        properties = feature[
            "properties"
        ]

        rows.append({
            "cell_id": properties["cell_id"],
            "vegetation_fraction": (
                properties["mean"]
                if properties.get("mean")
                is not None
                else 0
            )
        })

    return pd.DataFrame(rows)


GRID_VEGETATION_PATH = (
    PROCESSED_DIR
    / "grid_vegetation_fraction.parquet"
)

if GRID_VEGETATION_PATH.exists():
    grid_vegetation = pd.read_parquet(
        GRID_VEGETATION_PATH
    )
else:
    grid_vegetation = (
        calculate_grid_vegetation_fraction(
            grid_wgs84,
            vegetation_mask
        )
    )

    grid_vegetation.to_parquet(
        GRID_VEGETATION_PATH,
        index=False
    )

vegetation_fraction_lookup = dict(
    zip(
        grid_vegetation["cell_id"],
        grid_vegetation["vegetation_fraction"]
    )
)

grid_vegetation[
    "vegetation_fraction"
].describe()

## 8. Performance and production decisions

The main successful optimization was batching NDVI/NDMI statistics for all good acquisitions into **one Earth Engine request per cell-year**.

For `PT_2087 × 2019`:

- naive complete cell-year implementation: approximately **87.6 s**;
- optimized complete cell-year implementation: approximately **8.2 s**;
- all NDVI, NDMI and quality differences between old and batched implementations: **0.0**.

Profiling showed that daily SCL quality computation became the remaining bottleneck.

Several further strategies were tested and rejected because they did not improve the total workload enough to justify the extra complexity:

- multi-year processing in one Earth Engine graph;
- multiple cells in parallel threads;
- spatial `reduceRegions()` batching;
- stacked daily quality bands;
- coarser SCL quality resolutions.

The SCL quality calculation remains at its native **20 m** resolution. In the benchmark it was also the fastest tested scale:

| SCL quality scale | Time |
|---:|---:|
| **20 m** | **4.36 s** |
| 40 m | 5.63 s |
| 60 m | 8.96 s |
| 100 m | 6.91 s |

The final production unit therefore remains **one cell × one year**, processed sequentially with checkpoints.

## 9. Checkpoints, retries and recoverable extraction

Long national extraction runs should never depend on one uninterrupted notebook session.

Each completed `cell_id × year` is therefore written immediately to its own Parquet file. The runner:

- skips existing files on restart;
- writes through a temporary file and atomically renames it;
- retries transient Earth Engine errors;
- records failures without losing completed work.

For long-running execution, the same logic is intended to run from `scripts/extract_sentinel.py` on a persistent VM rather than keeping this notebook open.

In [ ]:
SENTINEL_PARTS_DIR = (
    PROCESSED_DIR
    / "sentinel_weekly_parts"
)

SENTINEL_PARTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def get_cell_year_output_path(
    cell_id,
    year
):
    return (
        SENTINEL_PARTS_DIR
        / f"{cell_id}_{year}.parquet"
    )


def save_cell_year_result(
    weekly,
    cell_id,
    year
):
    output_path = (
        get_cell_year_output_path(
            cell_id,
            year
        )
    )

    temp_path = (
        output_path.with_suffix(
            ".tmp.parquet"
        )
    )

    weekly.to_parquet(
        temp_path,
        index=False
    )

    temp_path.replace(
        output_path
    )

    return output_path

In [ ]:
def process_cell_year_with_retry(
    cell_id,
    year,
    vegetation_fraction,
    max_attempts=3
):
    for attempt in range(
        1,
        max_attempts + 1
    ):
        try:
            return process_cell_year_final(
                cell_id,
                year,
                vegetation_fraction
            )

        except Exception as error:
            if attempt == max_attempts:
                raise

            wait_seconds = 10 * attempt

            print(
                f"Retry {attempt}/"
                f"{max_attempts - 1} "
                f"for {cell_id} {year}"
            )
            print(
                f"Waiting {wait_seconds}s "
                f"after: {error}"
            )

            time.sleep(
                wait_seconds
            )

In [ ]:
def run_sentinel_extraction(
    cell_ids,
    years
):
    total = (
        len(cell_ids)
        * len(years)
    )

    completed = 0
    skipped = 0
    failed = []

    for cell_id in cell_ids:
        vegetation_fraction = (
            vegetation_fraction_lookup[
                cell_id
            ]
        )

        for year in years:
            completed += 1

            output_path = (
                get_cell_year_output_path(
                    cell_id,
                    year
                )
            )

            if output_path.exists():
                skipped += 1
                print(
                    f"[{completed}/{total}] "
                    f"SKIP {cell_id} {year}"
                )
                continue

            try:
                start = (
                    time.perf_counter()
                )

                weekly = (
                    process_cell_year_with_retry(
                        cell_id,
                        year,
                        vegetation_fraction
                    )
                )

                save_cell_year_result(
                    weekly,
                    cell_id,
                    year
                )

                elapsed = (
                    time.perf_counter()
                    - start
                )

                print(
                    f"[{completed}/{total}] "
                    f"DONE {cell_id} {year} "
                    f"({elapsed:.1f}s)"
                )

            except Exception as error:
                failed.append({
                    "cell_id": cell_id,
                    "year": year,
                    "error": str(error)
                })

                print(
                    f"[{completed}/{total}] "
                    f"FAILED {cell_id} {year}: "
                    f"{error}"
                )

    print("\nFinished.")
    print("Skipped:", skipped)
    print("Failed:", len(failed))

    return pd.DataFrame(
        failed
    )

### 9.1 Production validation

The checkpoint runner was validated before national execution.

A five-cell × two-year test considered 10 cell-years:

- 2 existing checkpoints were correctly skipped;
- 8 new cell-years were processed;
- **0 failures** occurred;
- observed processing times were approximately 5.8–10.3 seconds per new cell-year.

A restart test also confirmed that previously written files are immediately skipped.

This makes the extraction recoverable across notebook restarts, SSH disconnects or VM restarts.

## 10. Export static inputs for the production script

The VM / production script only needs the final grid and the cached vegetation fractions as local static inputs. Sentinel imagery and CORINE remain accessed through Earth Engine.

In [ ]:
GRID_PROCESSED_PATH = (
    PROCESSED_DIR
    / "grid_5km.parquet"
)

grid.to_parquet(
    GRID_PROCESSED_PATH,
    index=False
)

grid_vegetation.to_parquet(
    GRID_VEGETATION_PATH,
    index=False
)

print(
    GRID_PROCESSED_PATH,
    "->",
    GRID_PROCESSED_PATH.exists()
)
print(
    GRID_VEGETATION_PATH,
    "->",
    GRID_VEGETATION_PATH.exists()
)

## 11. Final feature schema

Each weekly Sentinel row contains **17 columns**:

| Group | Columns |
|---|---|
| Identification | `cell_id`, `reference_date` |
| Static support | `vegetation_fraction` |
| Observation metadata | `selected_date`, `image_age_days`, `valid_pct` |
| NDVI current state | `NDVI_mean`, `NDVI_median`, `NDVI_p10`, `NDVI_p90` |
| NDMI current state | `NDMI_mean`, `NDMI_median`, `NDMI_p10`, `NDMI_p90` |
| 60-day trend | `vegetation_obs_60d`, `NDVI_slope_30d`, `NDMI_slope_30d` |

### Frozen methodological decisions

- Sentinel source: `COPERNICUS/S2_SR_HARMONIZED`
- quality source: SCL at 20 m
- quality denominator: static CORINE vegetation-support pixels
- invalid SCL: 0, 1, 3, 8, 9, 10, 11
- acquisition unit: cell × physical acquisition day
- weekly policy: 7d ≥95%, 14d ≥90%, 30d ≥85%
- current-state statistics: NDVI / NDMI mean, median, P10, P90
- trend: all distinct acquisitions ≥85% valid in previous 60 days
- minimum trend observations: 3
- slope units: index change per 30 days
- processing unit: one cell × one year
- production strategy: sequential processing with atomic checkpoints and retries

The next production task is to generate the 2017–2025 Sentinel weekly parts and then join them to the common `cell_id × reference_date` modelling table.